In [1]:
%load_ext autoreload
%autoreload 2

import os
import glob
import cv2
import numpy as np

from SFace_utils import initialize_model, load_face_templates, identify_face_from_memory, extract_all_faces, files_from_folder, \
                        ideentify_faces

Operational Guidance for Low-Light & Motion
- Lighting Optimization: Facial verification algorithms degrade quickly in backlit setups (e.g., subject standing directly in front of a window). Diffuse, front-facing lighting produces the most distinct cosine separation.
- Tuning VIDEO_SFACE_THRESHOLD:
    - Raise to 0.363 if false acceptances (giving access to the wrong person) carry heavy consequences.
    - Lower to 0.30–0.32 if camera sensor noise or low frame sharpness causes frequent false rejections ("Unknown").

In [14]:
# Set path
# recognizer_model_path = '../Datasets/face_recognition_sface_2021dec.onnx'
recognizer_model_path = '../models/face_recognition_sface_2021dec.onnx'
# detector_model_path = '../Datasets/face_detection_yunet_2023mar.onnx'
detector_model_path = '../models/face_detection_yunet_2023mar.onnx'
output_folder = '../datasets/extracted_faces'
test_img_path = '../Datasets/group_photo.jpg'
dataset_dir = '../database'

input_size=(0, 0) 
score_threshold=0.45
nms_threshold=0.3

In [7]:
detector_model, recognizer_model = initialize_model(detector_model_path=detector_model_path, recognizer_model_path=recognizer_model_path, 
                        input_size=input_size, score_threshold=score_threshold, nms_threshold=nms_threshold)

In [8]:
# 1. Load the dataset from directory at runtime
face_database_dict = load_face_templates(detector_model=detector_model, recognizer_model=recognizer_model, 
                                         dataset_dir=dataset_dir)

Indexing dataset folder: ../database
 -> Indexed: Animesh (4 photos)
 -> Indexed: Sunny Singh (5 photos)
 -> Indexed: Tejrit Tyagi (5 photos)
 -> Indexed: Nivan Tyagi (5 photos)
 -> Indexed: Vimala (4 photos)
 -> Indexed: Shrinav Tyagi (5 photos)
 -> Indexed: Rakesh Ranjan (6 photos)
--- Indexing Complete ---



In [9]:
# 2. Test identification
identify_face_from_memory(detector_model=detector_model, recognizer_model=recognizer_model, 
                          query_img_path="../Datasets/vimala.jpg", templates=face_database_dict)

Result: Match found -> Vimala (Score: 0.9286): ../Datasets/vimala.jpg


'Vimala'

In [10]:
test_images = ['Tejrit.jpg', 'Shrinav.jpg', 'vimala.jpg', 'obama.jpg', 'group_photo.jpg']

for idx in test_images:
    # identify_face_from_memory("Datasets/"+idx, face_database_dict)
    identify_face_from_memory(detector_model=detector_model, recognizer_model=recognizer_model, 
                          query_img_path="../Datasets/"+idx, templates=face_database_dict)

Result: Match found -> Tejrit Tyagi (Score: 0.8793): ../Datasets/Tejrit.jpg
Result: Match found -> Shrinav Tyagi (Score: 0.8420): ../Datasets/Shrinav.jpg
Result: Match found -> Vimala (Score: 0.9286): ../Datasets/vimala.jpg
Result: Match found -> Unknown (Score: 0.1723): ../Datasets/obama.jpg
Result: Match found -> Nivan Tyagi (Score: 0.8030): ../Datasets/group_photo.jpg


In [15]:
# Extract all face from images
extract_all_faces(image_path=test_img_path, recognizer_model=recognizer_model, 
                  detector_model=detector_model, output_folder=output_folder, score_threshold=0.3)

Detected 5 face(s). Extracting clean crops...
 -> Saved: ../datasets/extracted_faces/face_1.jpg
 -> Saved: ../datasets/extracted_faces/face_2.jpg
 -> Saved: ../datasets/extracted_faces/face_3.jpg
 -> Saved: ../datasets/extracted_faces/face_4.jpg
 -> Saved: ../datasets/extracted_faces/face_5.jpg

Finished! Check the '../datasets/extracted_faces' directory for your images.


5

In [16]:
# dataset_path = "../raw_data/images"
dataset_path = "../datasets/images"

file_list = files_from_folder(dataset_path=dataset_path)
file_list

Found 0 images across all subfolders.


[]

In [11]:
# Extract all face from images
file_count = 0 # to control file index for file name over directory images

for idx in file_list:
    file_count = extract_all_faces(image_path=idx, recognizer_model=recognizer_model, detector_model=detector_model, 
                    output_folder="../raw_data/extracted_faces", score_threshold=0.7, file_count=file_count)

Detected 1 face(s). Extracting clean crops...
 -> Saved: ../raw_data/extracted_faces/face_1.jpg

Finished! Check the '../raw_data/extracted_faces' directory for your images.
Detected 1 face(s). Extracting clean crops...
 -> Saved: ../raw_data/extracted_faces/face_2.jpg

Finished! Check the '../raw_data/extracted_faces' directory for your images.
Detected 1 face(s). Extracting clean crops...
 -> Saved: ../raw_data/extracted_faces/face_3.jpg

Finished! Check the '../raw_data/extracted_faces' directory for your images.
Detected 1 face(s). Extracting clean crops...
 -> Saved: ../raw_data/extracted_faces/face_4.jpg

Finished! Check the '../raw_data/extracted_faces' directory for your images.


In [7]:
dataset_path = "Datasets/extracted_faces/"

file_list = files_from_folder(dataset_path=dataset_path)
file_list

Found 5 images across all subfolders.
Datasets/extracted_faces/face_3.jpg
Datasets/extracted_faces/face_5.jpg
Datasets/extracted_faces/face_1.jpg
Datasets/extracted_faces/face_2.jpg
Datasets/extracted_faces/face_4.jpg


['Datasets/extracted_faces/face_3.jpg',
 'Datasets/extracted_faces/face_5.jpg',
 'Datasets/extracted_faces/face_1.jpg',
 'Datasets/extracted_faces/face_2.jpg',
 'Datasets/extracted_faces/face_4.jpg']

In [9]:
ideentify_faces(detector_model=detector_model, recognizer_model=recognizer_model, face_database_dict=face_database_dict, 
                images_list=file_list)

Result: Match found -> Vimala (Score: 0.6950): Datasets/extracted_faces/face_3.jpg
Result: Match found -> Unknown (Score: 0.2078): Datasets/extracted_faces/face_5.jpg
Result: Match found -> Nivan Tyagi (Score: 0.8217): Datasets/extracted_faces/face_1.jpg
Result: Match found -> Unknown (Score: 0.2313): Datasets/extracted_faces/face_2.jpg
Result: Match found -> Shrinav Tyagi (Score: 0.7989): Datasets/extracted_faces/face_4.jpg
